# Day 038 — Exercise 5: eda_report

**What you'll build:** `eda_report(df) -> dict` — a single-call EDA that returns shape, null counts, numeric describe-style statistics, category value counts, and a correlation matrix, all in one structured dict.

**Why it matters:** In practice you need a quick snapshot of any new dataset in seconds. A report dict is loggable, serialisable to JSON, and can be diffed between runs to spot data drift.

## Provided: All Four EDA Helpers

In [ ]:
import pandas as pd

def distribution_summary(df: pd.DataFrame, col: str) -> dict:
    s = df[col]
    if pd.api.types.is_numeric_dtype(s):
        return {
            'count':      int(s.count()),
            'mean':       round(float(s.mean()), 4),
            'std':        round(float(s.std()), 4),
            'min':        float(s.min()),
            'q25':        float(s.quantile(0.25)),
            'median':     float(s.quantile(0.50)),
            'q75':        float(s.quantile(0.75)),
            'max':        float(s.max()),
            'null_count': int(s.isnull().sum()),
        }
    counts = s.value_counts()
    return {
        'count':      int(s.count()),
        'unique':     int(s.nunique()),
        'top':        str(counts.index[0]) if len(counts) else None,
        'top_freq':   int(counts.iloc[0])  if len(counts) else 0,
        'null_count': int(s.isnull().sum()),
    }


import pandas as pd

def top_groups(df: pd.DataFrame, group_col: str, value_col: str,
               n: int = 5) -> pd.DataFrame:
    return (
        df.groupby(group_col)[value_col]
        .agg(total='sum', mean='mean', count='count')
        .reset_index()
        .nlargest(n, 'total')
        .reset_index(drop=True)
    )


import pandas as pd

def correlation_summary(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    corr   = df.select_dtypes(include='number').corr()[target_col].drop(target_col)
    result = pd.DataFrame({'feature': corr.index.tolist(), 'correlation': corr.values})
    result['_abs'] = result['correlation'].abs()
    result = result.sort_values('_abs', ascending=False).drop(columns='_abs')
    return result.reset_index(drop=True)


import pandas as pd

def pivot_summary(df: pd.DataFrame, index: str, columns: str,
                  values: str, aggfunc: str = 'mean') -> pd.DataFrame:
    piv = pd.pivot_table(
        df, values=values, index=index, columns=columns,
        aggfunc=aggfunc, fill_value=0,
    )
    piv.columns.name = None
    return piv.reset_index()

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

# SALES_DF: 8 rows, 6 columns
# product:  Widget×4, Gadget×2, Doohickey×2
# revenue   = price × quantity  (pre-computed)
SALES_DF = pd.DataFrame({
    'product':  ['Widget', 'Widget', 'Widget', 'Widget',
                 'Gadget', 'Gadget', 'Doohickey', 'Doohickey'],
    'category': ['Elec', 'Elec', 'Elec', 'Elec',
                 'Elec', 'Elec', 'Access', 'Access'],
    'region':   ['North', 'South', 'East', 'West',
                 'North', 'East', 'North', 'South'],
    'price':    [25.0, 25.0, 25.0, 25.0, 150.0, 150.0, 8.0, 8.0],
    'quantity': [10, 5, 4, 6, 3, 7, 50, 15],
    'revenue':  [250.0, 125.0, 100.0, 150.0, 450.0, 1050.0, 400.0, 120.0],
})

## Your Implementation

In [ ]:
def eda_report(df: pd.DataFrame) -> dict:
    """
    Run a full EDA on df and return a structured dict.

    Returns:
        shape            — (rows, cols) tuple
        null_counts      — {col: null_count} dict
        numeric_summary  — {col: {stat: val}} from describe().round(2)
        category_counts  — {col: {value: count}} from value_counts()
        correlations     — {col: {col: pearson_r}} from corr().round(4)
    """
    num_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    # TODO: return dict with shape, null_counts, numeric_summary,
    #       category_counts, correlations
    # Hint: df[num_cols].describe().round(2).to_dict() for numeric_summary
    # Hint: {col: df[col].value_counts().to_dict() for col in cat_cols}
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined, returns dict
    try:
        assert 'eda_report' in globals()
        result = eda_report(SALES_DF)
        assert isinstance(result, dict), \
            f'expected dict, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 1: returns a dict')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: all 5 top-level keys present
    try:
        result = eda_report(SALES_DF)
        for k in ('shape', 'null_counts', 'numeric_summary',
                  'category_counts', 'correlations'):
            assert k in result, f'missing key: {k!r}'
        passed += 1; print('\u2705 Check 2: all 5 keys present')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: shape matches df.shape
    try:
        result = eda_report(SALES_DF)
        assert result['shape'] == SALES_DF.shape, \
            f'shape={result["shape"]}, expected {SALES_DF.shape}'
        passed += 1; print(f'\u2705 Check 3: shape={result["shape"]}')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: numeric_summary contains describe stats for numeric cols
    try:
        result = eda_report(SALES_DF)
        ns = result['numeric_summary']
        assert isinstance(ns, dict), 'numeric_summary should be a dict'
        for col in ('price', 'quantity', 'revenue'):
            assert col in ns, f'numeric_summary missing column: {col}'
        # describe() returns count, mean, std, min, 25%, 50%, 75%, max
        assert 'mean' in ns['revenue'], \
            f'numeric_summary[revenue] missing mean; keys={list(ns["revenue"].keys())}'
        passed += 1; print('\u2705 Check 4: numeric_summary has price/quantity/revenue with stats')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: category_counts has entries for string columns
    try:
        result = eda_report(SALES_DF)
        cc = result['category_counts']
        assert isinstance(cc, dict), 'category_counts should be a dict'
        for col in ('product', 'category', 'region'):
            assert col in cc, f'category_counts missing column: {col}'
        assert cc['product']['Widget'] == 4, \
            f'Widget count={cc["product"]["Widget"]}, expected 4'
        passed += 1; print("\u2705 Check 5: category_counts has product/category/region; Widget=4")
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import pandas as pd

def eda_report(df: pd.DataFrame) -> dict:
    num_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    return {
        'shape':           df.shape,
        'null_counts':     df.isnull().sum().to_dict(),
        'numeric_summary': df[num_cols].describe().round(2).to_dict() if num_cols else {},
        'category_counts': {col: df[col].value_counts().to_dict() for col in cat_cols},
        'correlations':    df[num_cols].corr().round(4).to_dict() if len(num_cols) > 1 else {},
    }
```

</details>